# M01-P01 — Exploring Bedrock LLMs

En esta práctica vas a trabajar directamente con **Amazon Bedrock Converse API**, sin LangChain ni otras abstracciones.

El objetivo no es escribir mucho código, sino **abrir la caja** y observar qué enviamos al modelo, qué devuelve y cómo cambian los resultados al modificar:

- `system prompt`;
- temperatura;
- longitud máxima de salida;
- contexto proporcionado;
- streaming;
- modelo, si el instructor habilita un segundo modelo.

> Requisito: haber completado P00. Si la sesión SSO ha caducado, ejecuta en terminal:
>
> `aws sso login --profile genai-course --use-device-code`


## 1. Preparar el cliente y comprobar la identidad

Esta celda usa el perfil `genai-course` configurado en P00. No hay claves AWS dentro del notebook.


In [ ]:
import json
import os
import time

import boto3
import pandas as pd

from enterprise_genai.config import get_settings

settings = get_settings()

session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=settings.aws_region,
)

identity = session.client('sts').get_caller_identity()
runtime = session.client('bedrock-runtime', region_name=settings.aws_region)

MODEL_ID = settings.bedrock_text_model_id

print('AWS identity:', identity['Arn'])
print('Region:', settings.aws_region)
print('Model:', MODEL_ID)


## 2. Primera llamada: ver la petición y la respuesta

Primero construimos explícitamente la petición. Observa las cuatro piezas principales:

1. `modelId`: qué modelo o inference profile vamos a usar.
2. `system`: instrucciones de comportamiento.
3. `messages`: contenido de la conversación.
4. `inferenceConfig`: parámetros que controlan la generación.


In [ ]:
incident = (
    'Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. '
    'El servicio web corporativo funciona correctamente. '
    'El equipo de red informa de errores intermitentes en el backend de autenticación.'
)

request = {
    'modelId': MODEL_ID,
    'system': [
        {
            'text': (
                'Eres un asistente de soporte empresarial. '
                'Resume incidencias de forma clara y no inventes datos.'
            )
        }
    ],
    'messages': [
        {
            'role': 'user',
            'content': [
                {
                    'text': f'Resume esta incidencia en 3 viñetas:\n\n{incident}'
                }
            ],
        }
    ],
    'inferenceConfig': {
        'temperature': 0.2,
        'maxTokens': 200,
    },
}

print(json.dumps(request, indent=2, ensure_ascii=False))


In [ ]:
response = runtime.converse(**request)

text = response['output']['message']['content'][0]['text']

print(text)
print('\nStop reason:', response.get('stopReason'))
print('Usage:', response.get('usage'))
print('Metrics:', response.get('metrics'))


### Qué debes observar

- `inputTokens`: tokens enviados al modelo.
- `outputTokens`: tokens generados.
- `totalTokens`: suma de ambos.
- `latencyMs`: tiempo empleado por Bedrock.
- `stopReason`: por qué terminó la generación.

Estas métricas nos permitirán hablar más adelante de **coste, rendimiento y observabilidad** usando datos reales.


## 3. Un pequeño banco de pruebas

La siguiente función no oculta la llamada a Bedrock: simplemente evita repetir código mientras hacemos experimentos. Devuelve texto, tokens, latencia y motivo de parada.


In [ ]:
def run_experiment(
    label,
    user_message,
    *,
    system_prompt=None,
    temperature=0.2,
    max_tokens=200,
    model_id=MODEL_ID,
):
    kwargs = {
        'modelId': model_id,
        'messages': [
            {
                'role': 'user',
                'content': [{'text': user_message}],
            }
        ],
        'inferenceConfig': {
            'temperature': temperature,
            'maxTokens': max_tokens,
        },
    }

    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]

    started = time.perf_counter()
    response = runtime.converse(**kwargs)
    elapsed_ms = round((time.perf_counter() - started) * 1000)

    text = response['output']['message']['content'][0]['text']
    usage = response.get('usage', {})
    metrics = response.get('metrics', {})

    return {
        'label': label,
        'text': text,
        'input_tokens': usage.get('inputTokens'),
        'output_tokens': usage.get('outputTokens'),
        'bedrock_latency_ms': metrics.get('latencyMs'),
        'client_elapsed_ms': elapsed_ms,
        'stop_reason': response.get('stopReason'),
    }


## 4. Experimento A — ¿Para qué sirve el system prompt?

Usamos exactamente la misma incidencia con dos comportamientos distintos. No cambiamos los datos; cambiamos el **rol y las restricciones**.


In [ ]:
user_message = f'Analiza esta incidencia y dime qué harías:\n\n{incident}'

system_variants = [
    ('Sin system prompt', None),
    (
        'Coordinador NOC',
        (
            'Actúa como coordinador de un NOC empresarial. '
            'Responde con: impacto, hipótesis y siguiente acción. '
            'No afirmes una causa raíz si no está demostrada.'
        ),
    ),
]

system_results = []

for label, system_prompt in system_variants:
    result = run_experiment(
        label,
        user_message,
        system_prompt=system_prompt,
        temperature=0.2,
        max_tokens=220,
    )
    system_results.append(result)
    print(f'\n--- {label} ---')
    print(result['text'])

pd.DataFrame(system_results)[
    ['label', 'input_tokens', 'output_tokens', 'bedrock_latency_ms', 'stop_reason']
]


**Modifica** el segundo system prompt. Por ejemplo, conviértelo en:

- un ejecutivo que solo quiere impacto de negocio;
- un técnico de nivel 1;
- un responsable de seguridad.

Vuelve a ejecutar la celda y compara cómo cambia la respuesta.


## 5. Experimento B — Temperatura

La temperatura controla cuánto se favorece una generación más predecible o más variada. No significa que `temperature=0` garantice matemáticamente la misma salida en todos los casos.

Usaremos una tarea donde la variación sea visible: proponer títulos para una incidencia.


In [ ]:
temperature_results = []

for temperature in [0.0, 0.8]:
    result = run_experiment(
        f'temperature={temperature}',
        'Propón 5 títulos breves y profesionales para esta incidencia:\n\n' + incident,
        system_prompt='Eres un redactor técnico de un centro de operaciones.',
        temperature=temperature,
        max_tokens=180,
    )
    temperature_results.append(result)
    print(f'\n--- temperature={temperature} ---')
    print(result['text'])

pd.DataFrame(temperature_results)[
    ['label', 'input_tokens', 'output_tokens', 'bedrock_latency_ms']
]


**Prueba** también un valor intermedio, por ejemplo `0.4`, y observa si el cambio es evidente en esta tarea.

La idea no es encontrar una temperatura universalmente correcta: depende del caso de uso.


## 6. Experimento C — `maxTokens` y respuestas truncadas

Ahora mantenemos prompt y temperatura y cambiamos únicamente el límite de salida.


In [ ]:
length_results = []

for max_tokens in [40, 220]:
    result = run_experiment(
        f'maxTokens={max_tokens}',
        (
            'Redacta un informe de incidencia con las secciones: '
            'Resumen, Impacto, Evidencias, Hipótesis y Próximos pasos.\n\n'
            + incident
        ),
        system_prompt='No inventes información que no aparezca en la incidencia.',
        temperature=0.2,
        max_tokens=max_tokens,
    )
    length_results.append(result)
    print(f'\n--- maxTokens={max_tokens} ---')
    print(result['text'])
    print('Stop reason:', result['stop_reason'])

pd.DataFrame(length_results)[
    ['label', 'output_tokens', 'bedrock_latency_ms', 'stop_reason']
]


Fíjate especialmente en `stop_reason`. Si el límite es demasiado pequeño, la salida puede terminar porque se alcanzó la longitud máxima y no porque el modelo hubiera terminado naturalmente.


## 7. Experimento D — El contexto cambia lo que el modelo puede responder

Todavía **no estamos haciendo RAG**. Aquí añadimos manualmente un fragmento de información empresarial para comprobar la diferencia entre:

- preguntar sin contexto corporativo;
- proporcionar explícitamente datos internos en el prompt.

Este experimento prepara el terreno para embeddings y RAG en módulos posteriores.


In [ ]:
question = '¿Cuál es el SLA de respuesta para una incidencia P1 de Asteria Retail en Managed VPN?'

corporate_context = '''
Contrato CTR001
Cliente: Asteria Retail
Servicio: Managed VPN
Nivel de cliente: premium
SLA de respuesta para P1: 30 minutos
'''

without_context = run_experiment(
    'Sin contexto',
    question,
    system_prompt=(
        'Responde solo si dispones de información suficiente. '
        'Si no la tienes, dilo claramente.'
    ),
    temperature=0,
    max_tokens=100,
)

with_context = run_experiment(
    'Con contexto',
    f'Contexto corporativo:\n{corporate_context}\n\nPregunta:\n{question}',
    system_prompt='Responde únicamente a partir del contexto corporativo proporcionado.',
    temperature=0,
    max_tokens=100,
)

for result in [without_context, with_context]:
    print(f"\n--- {result['label']} ---")
    print(result['text'])

pd.DataFrame([without_context, with_context])[
    ['label', 'input_tokens', 'output_tokens', 'bedrock_latency_ms']
]


Observa que proporcionar contexto aumenta los tokens de entrada. Más adelante automatizaremos la selección de ese contexto con búsqueda semántica y RAG.


## 8. Streaming — ver cómo llega la respuesta

Hasta ahora hemos esperado a que la respuesta estuviera completa. `converse_stream()` permite procesar los fragmentos conforme llegan.

Observa los eventos del stream:

- `contentBlockDelta`: fragmentos de texto;
- `messageStop`: motivo de finalización;
- `metadata`: uso de tokens y métricas.


In [ ]:
stream_response = runtime.converse_stream(
    modelId=MODEL_ID,
    system=[{'text': 'Eres un asistente de soporte. Responde con claridad y brevedad.'}],
    messages=[
        {
            'role': 'user',
            'content': [
                {
                    'text': (
                        'Explica en un párrafo por qué una caída de autenticación '
                        'puede afectar a una VPN aunque la web corporativa siga disponible.'
                    )
                }
            ],
        }
    ],
    inferenceConfig={'temperature': 0.2, 'maxTokens': 180},
)

stream_usage = {}
stream_metrics = {}
stream_stop_reason = None

for event in stream_response['stream']:
    if 'contentBlockDelta' in event:
        delta = event['contentBlockDelta'].get('delta', {})
        if 'text' in delta:
            print(delta['text'], end='', flush=True)
    elif 'messageStop' in event:
        stream_stop_reason = event['messageStop'].get('stopReason')
    elif 'metadata' in event:
        stream_usage = event['metadata'].get('usage', {})
        stream_metrics = event['metadata'].get('metrics', {})

print('\n\nStop reason:', stream_stop_reason)
print('Usage:', stream_usage)
print('Metrics:', stream_metrics)


## 9. Mini-reto — modifica, no reescribas

Elige **una** de estas modificaciones y vuelve a ejecutar los experimentos:

1. Cambia la incidencia por otra situación empresarial.
2. Haz que el system prompt obligue a separar hechos de hipótesis.
3. Busca un `maxTokens` suficientemente pequeño para provocar truncamiento y luego corrígelo.
4. Compara `temperature=0`, `0.4` y `0.8` en una tarea creativa.

Anota qué parámetro has cambiado y qué efecto observable has encontrado.


In [ ]:
conclusion = (
    'Parámetro modificado: \n'
    'Efecto observado: \n'
    'Cuándo usaría este ajuste: \n'
)

print(conclusion)


## 10. Reto opcional — comparar modelos

El curso puede habilitar un segundo modelo de forma controlada. Si el instructor define `BEDROCK_TEXT_MODEL_ID_ALT`, esta celda ejecuta la misma petición con ambos modelos y compara tokens y latencia.

Si no existe un segundo modelo autorizado, la celda simplemente lo indica y no falla.


In [ ]:
ALT_MODEL_ID = os.getenv('BEDROCK_TEXT_MODEL_ID_ALT', '').strip()

if not ALT_MODEL_ID:
    print('No hay un segundo modelo habilitado. Reto opcional omitido.')
else:
    comparison = []
    for model_id in [MODEL_ID, ALT_MODEL_ID]:
        result = run_experiment(
            model_id,
            f'Resume esta incidencia en 3 viñetas:\n\n{incident}',
            system_prompt='No inventes datos.',
            temperature=0.2,
            max_tokens=160,
            model_id=model_id,
        )
        comparison.append(result)
        print(f'\n--- {model_id} ---')
        print(result['text'])

    display(pd.DataFrame(comparison)[
        ['label', 'input_tokens', 'output_tokens', 'bedrock_latency_ms', 'client_elapsed_ms']
    ])


## Resultado de la práctica

Has realizado llamadas reales a un LLM desde Python y has observado directamente:

- estructura de una petición Converse;
- system prompt y mensaje de usuario;
- temperatura;
- límite de tokens;
- contexto;
- streaming;
- consumo de tokens;
- latencia;
- motivo de finalización.

En P02 utilizaremos esta misma capa de acceso para construir un **analizador de incidencias con salida estructurada**.
